# Leave-One-Site-Out Hospital Generalization


In [1]:
import os
import sys
from pathlib import Path

root_dir = Path("..").resolve() if Path(".").resolve().name == "notebooks" else Path(".").resolve()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import pandas as pd
from src.data import load_all_sites
from src.fl import run_federated

config = {
    "dataset": "fed_heart_disease",
    "seeds": list(range(42, 52)),
    "rounds": 50,
    "local_epochs": 1,
    "lr": 0.001,
    "batch_size": 4,
    "e1_csv": str(root_dir / "results" / "e1.csv"),
    "output_csv": str(root_dir / "results" / "e2.csv"),
}

Loading hospital datasets across all 4 sites.

In [2]:
sites_data = load_all_sites()
print("Loaded 4 hospital site datasets.")

Loaded 4 hospital site datasets.


Training FedAvg leaving out one site at a time across 10 random seeds, evaluating on the held-out unseen site.

In [3]:
records = []
for h in range(4):
    print(f"Training FedAvg without site {h}...")
    train_sites = {s: sites_data[s] for s in range(4) if s != h}
    recs = run_federated(
        train_sites, seeds=config["seeds"], rounds=config["rounds"],
        local_epochs=config["local_epochs"], lr=config["lr"],
        batch_size=config["batch_size"], exp="e2", heldout_site=h,
        method="fedavg_loso", eval_sites=sites_data
    )
    records.extend(recs)

df_e2 = pd.DataFrame(records)
os.makedirs(os.path.dirname(config["output_csv"]), exist_ok=True)
df_e2.to_csv(config["output_csv"], index=False)
print(f"Saved {len(df_e2)} rows to {config['output_csv']}")

Training FedAvg without site 0...


Training FedAvg without site 1...


Training FedAvg without site 2...


Training FedAvg without site 3...


Saved 8000 rows to C:\Users\Soumadipta Konar\Desktop\fl-hospital-consortium\results\e2.csv


Comparison table: generalization of the federated model on an unseen hospital versus the hospital training alone (local-only).

In [4]:
df_e1 = pd.read_csv(config["e1_csv"])
local_final = df_e1[(df_e1["method"] == "local-only") & (df_e1["round"] == config["rounds"])].copy()
local_final["site"] = local_final["site"].astype(int)
local_summary = local_final.groupby("site")["metric_value"].agg(["mean", "std"]).reset_index()

loso_final = df_e2[(df_e2["round"] == config["rounds"]) & (df_e2["site"] == df_e2["heldout_site"])].copy()
loso_final["site"] = loso_final["site"].astype(int)
loso_summary = loso_final.groupby("site")["metric_value"].agg(["mean", "std"]).reset_index()

comparison = pd.merge(local_summary, loso_summary, on="site", suffixes=("_local", "_loso"))
comparison["local_only"] = comparison.apply(lambda r: f"{r['mean_local']:.3f} +/- {r['std_local']:.3f}", axis=1)
comparison["heldout_fl"] = comparison.apply(lambda r: f"{r['mean_loso']:.3f} +/- {r['std_loso']:.3f}", axis=1)
comparison["delta (fl - local)"] = comparison["mean_loso"] - comparison["mean_local"]

site_names = {0: "cleveland", 1: "hungarian", 2: "switzerland", 3: "va"}
comparison["site_name"] = comparison["site"].map(site_names)

display_cols = ["site", "site_name", "local_only", "heldout_fl", "delta (fl - local)"]
print(comparison[display_cols].to_string(index=False))

 site   site_name      local_only      heldout_fl  delta (fl - local)
    0   cleveland 0.750 +/- 0.012 0.743 +/- 0.011           -0.006731
    1   hungarian 0.752 +/- 0.011 0.753 +/- 0.015            0.001124
    2 switzerland 0.775 +/- 0.148 0.669 +/- 0.051           -0.106250
    3          va 0.700 +/- 0.041 0.560 +/- 0.014           -0.140000
